# 금융 GraphRAG 후보 사람 검수

이 노트북은 `trial_v1` 시험 후보를 사람이 확인하는 화면입니다. 여기서 승인된 문항만 검수 결과로 저장되며, 이 노트북은 `golden_set_final.jsonl`을 자동으로 만들거나 확정하지 않습니다.

검수 순서: 후보 선택 → 질문·정답·근거 확인 → 승인/수정/제외/보류 → 검수 이력 저장

In [ ]:
from pathlib import Path
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
BATCH_DIR = PROJECT_ROOT / 'data/evaluation/candidates/trial_v5'
CHUNK_FILE = PROJECT_ROOT / 'data/evaluation/chunks/v1/chunks.jsonl'
GRAPH_REPORT = PROJECT_ROOT / 'data/evaluation/neo4j_aligned_sample/neo4j_mapping_report.json'

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
from services.golden_review import apply_review_action, append_review_history, write_reviewed_candidates

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

candidates = read_jsonl(BATCH_DIR / 'candidates.jsonl')
contexts = {row['chunk_id']: row for row in read_jsonl(CHUNK_FILE)}
graph_rows = {row['chunk_id']: row for row in json.loads(GRAPH_REPORT.read_text(encoding='utf-8')).get('rows', [])}
candidate_by_id = {row['candidate_id']: row for row in candidates}
history = []
print(f'후보 {len(candidates)}개를 불러왔습니다. 배치: {BATCH_DIR.name}')

In [ ]:
summary = pd.DataFrame([{
    'candidate_id': row['candidate_id'],
    'route': row['expected_route'],
    'difficulty': row['difficulty'],
    'hop_count': row['hop_count'],
    'critic': row['critic']['verdict'],
    'review_status': row['review_status'],
    'question': row['question'],
} for row in candidates])
display(summary)

In [ ]:
candidate_select = widgets.Dropdown(options=[row['candidate_id'] for row in candidates], description='후보')
question_box = widgets.Textarea(description='질문', layout=widgets.Layout(width='100%', height='80px'))
reference_box = widgets.Textarea(description='정답', layout=widgets.Layout(width='100%', height='120px'))
facts_box = widgets.Textarea(description='기대 사실', layout=widgets.Layout(width='100%', height='120px'))
detail_output = widgets.Output()

def show_candidate(candidate_id):
    row = candidate_by_id[candidate_id]
    question_box.value = row['question']
    reference_box.value = row['reference']
    facts_box.value = json.dumps(row['expected_facts'], ensure_ascii=False, indent=2)
    context_text = '\n\n'.join(contexts[cid]['text'] for cid in row['reference_context_ids'] if cid in contexts)
    relation_text = json.dumps(row['expected_relation_ids'], ensure_ascii=False, indent=2)
    source_text = json.dumps([{
        'chunk_id': contexts[cid].get('chunk_id'),
        'source_file': contexts[cid].get('source_file'),
        'source_path': contexts[cid].get('source_path'),
        'source_link': contexts[cid].get('source_link'),
    } for cid in row['reference_context_ids'] if cid in contexts], ensure_ascii=False, indent=2)
    with detail_output:
        clear_output(wait=True)
        display(Markdown(f"**유형:** `{row['expected_route']}`  |  **난이도:** `{row['difficulty']}`  |  **홉:** `{row['hop_count']}`  |  **판정 이유:** `{row.get('route_rationale', '')}`"))
        display(Markdown('### 관련 청크 원문\n```text\n' + context_text + '\n```'))
        display(Markdown('### 원본 출처\n```json\n' + source_text + '\n```'))
        display(Markdown('### 엔터티 ID\n```json\n' + json.dumps(row['expected_entity_ids'], ensure_ascii=False, indent=2) + '\n```'))
        display(Markdown('### Evidence ID\n```json\n' + json.dumps({'source_evidence_ids': row.get('source_evidence_ids', []), 'neo4j_evidence_ids': row.get('neo4j_evidence_ids', []), 'relation_identity_basis': row.get('relation_identity_basis')}, ensure_ascii=False, indent=2) + '\n```'))
        display(Markdown('### 그래프 경로\n```json\n' + relation_text + '\n```'))
        display(Markdown('### Critic 결과\n```json\n' + json.dumps(row['critic'], ensure_ascii=False, indent=2) + '\n```'))

candidate_select.observe(lambda change: show_candidate(change['new']) if change['name'] == 'value' else None)
display(candidate_select, question_box, reference_box, facts_box, detail_output)
show_candidate(candidate_select.value)

In [ ]:
approve_button = widgets.Button(description='승인', button_style='success')
revise_button = widgets.Button(description='수정 승인', button_style='warning')
reject_button = widgets.Button(description='제외', button_style='danger')
hold_button = widgets.Button(description='보류')
reviewer_box = widgets.Text(value='reviewer', description='검수자')
action_output = widgets.Output()

def save_action(action, revised=False):
    candidate_id = candidate_select.value
    row = candidate_by_id[candidate_id]
    revised_fields = None
    if revised:
        revised_fields = {
            'question': question_box.value,
            'reference': reference_box.value,
            'expected_facts': json.loads(facts_box.value),
        }
    with action_output:
        clear_output(wait=True)
        try:
            updated, event = apply_review_action(
                row, action=action, reviewer=reviewer_box.value,
                contexts=contexts, graph_rows=graph_rows, revised_fields=revised_fields
            )
            candidate_by_id[candidate_id] = updated
            candidates[:] = list(candidate_by_id.values())
            history.append(event)
            append_review_history(BATCH_DIR, event)
            write_reviewed_candidates(BATCH_DIR, candidates)
            display(Markdown(f"저장 완료: `{candidate_id}` → **{updated['review_status']}**"))
            show_candidate(candidate_id)
        except Exception as exc:
            display(Markdown(f"검수 저장 실패: `{type(exc).__name__}: {exc}`"))

approve_button.on_click(lambda _: save_action('approve'))
revise_button.on_click(lambda _: save_action('revise', revised=True))
reject_button.on_click(lambda _: save_action('reject'))
hold_button.on_click(lambda _: save_action('hold'))
display(reviewer_box, widgets.HBox([approve_button, revise_button, reject_button, hold_button]), action_output)

## 저장 파일

- `reviewed_candidates.jsonl`: 현재 후보별 검수 상태
- `review_history.jsonl`: 승인·수정·제외·보류 이력

수정 승인도 코드 기반 근거 검사를 다시 통과해야 저장됩니다. 최종 골든셋 파일은 별도 단계에서 승인 문항과 Neo4j 검증 상태를 확인한 뒤 생성합니다.